In [1]:
from pysr import PySRRegressor
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

# ----- Data loading -----------------------------------------------------------
data_dir = Path.cwd().parent / 'Data'
df = pd.read_hdf(data_dir / 'AppML_InitialProject_train.h5')
df = df[df['p_Truth_isElectron'] == 1].copy()
print(f"Loaded training set: {df.shape[0]} rows, {df.shape[1]} columns")

X = df.drop(columns=["p_Truth_Energy", "p_Truth_isElectron"])
y = df["p_Truth_Energy"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42,
)
print(f"Train: {X_train.shape[0]} rows, "
      f"Val: {X_val.shape[0]} rows, "
      f"Test: {X_test.shape[0]} rows")

# ----- Feature selection (XGB-informed) ---------------------------------------
top_features = [
    'pX_ecore',
    'pX_E3x5_Lr2',
    'p_pt_track',
    'pX_deltaPhi2',
    'p_eta',
    'p_ptcone40',
]


Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
Loaded training set: 37800 rows, 142 columns
Train: 26460 rows, Val: 5670 rows, Test: 5670 rows


In [ ]:

# ----- Subsample for tractable PySR runtime ----------------------------------
# 26k events × 300 iterations × population 40 is hours. Subsample to iterate fast.
# Use ~5k events for the first run, scale up if equations look promising.
N_SUBSAMPLE = 5000
rng = np.random.default_rng(42)
idx = rng.choice(len(X_train), size=N_SUBSAMPLE, replace=False)

X_pysr = X_train.iloc[idx][top_features].values
y_pysr = np.log(y_train.iloc[idx]).values

print(f"PySR fitting on {len(X_pysr)} events × {len(top_features)} features")
print(f"Target: log(E_true) in range [{y_pysr.min():.2f}, {y_pysr.max():.2f}]")

# ----- PySR setup -------------------------------------------------------------
model = PySRRegressor(
    niterations=300,
    populations=40,
    population_size=33,
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["log", "exp", "sqrt", "square", "abs"],
    maxsize=25,
    parsimony=0.0032,
    elementwise_loss="loss(prediction, target) = (prediction - target)^2",
    procs=4,
    model_selection="best",
    progress=True,
    random_state=42,
    deterministic=False,
    verbosity=1,
)

model.fit(X_pysr, y_pysr, variable_names=top_features)

# ----- Inspect the Pareto front -----------------------------------------------
print("\n=== Pareto front ===")
print(model.equations_[['complexity', 'loss', 'score', 'equation']])

print("\n=== Best equation (symbolic) ===")
print(model.sympy())

# ----- Evaluate on full val/test ----------------------------------------------
def relmad(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_pred - y_true) / y_true))

# Predict log(E) on full val/test, then exp() to recover linear E
log_pred_val  = model.predict(X_val[top_features].values)
log_pred_test = model.predict(X_test[top_features].values)

y_pred_val  = np.exp(log_pred_val)
y_pred_test = np.exp(log_pred_test)

print(f"\nRelMAD (val):  {relmad(y_val,  y_pred_val):.5f}")
print(f"RelMAD (test): {relmad(y_test, y_pred_test):.5f}")

# Also evaluate the simpler equations from the Pareto front for the writeup
print("\n=== RelMAD per equation on Pareto front ===")
for i, eq in enumerate(model.equations_.itertuples()):
    log_pred = model.predict(X_val[top_features].values, index=i)
    rm = relmad(y_val, np.exp(log_pred))
    print(f"  complexity={eq.complexity:2d}  loss={eq.loss:.4f}  RelMAD={rm:.4f}  "
          f"eq: {eq.equation}")

/Users/prometheus/Documents/Python/uni-env-3.13/lib/python3.13/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
Compiling Julia backend...


Loaded training set: 37800 rows, 142 columns
Train: 26460 rows, Val: 5670 rows, Test: 5670 rows
PySR fitting on 5000 events × 6 features
Target: log(E_true) in range [6.41, 13.68]


/Users/prometheus/Documents/Python/uni-env-3.13/lib/python3.13/site-packages/pysr/sr.py:1873: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 1.060e+05
Progress: 395 / 12000 total iterations (3.292%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           1.135e+00  0.000e+00  y = 10.656
2           8.498e-01  2.897e-01  y = log(pX_E3x5_Lr2)
4           5.817e-01  1.896e-01  y = log(pX_E3x5_Lr2 + p_pt_track)
5           4.772e-01  1.979e-01  y = abs(p_eta) + log(p_pt_track)
6           4.418e-01  7.719e-02  y = log(p_pt_track + (pX_E3x5_Lr2 * 0.53439))
7           3.334e-01  2.814e-01  y = log(p_pt_track + (p_pt_track * square(p_eta)))
9           3.334e-01  1.341e-05  y = log(p_pt_track + ((square(p_eta) * p_pt_track) - pX_de...
                                      ltaPhi2))
10          2.997e-01  1.065e-01  y = log(((0.68575 / (p_ptcone40 + 1.2523)) * pX_ecore) + p...
                       

[ Info: Final population:
[ Info: Results saved to:



=== Pareto front ===
    complexity      loss     score  \
0            1  1.135352  0.000000   
1            2  0.849831  0.289661   
2            4  0.581674  0.189564   
3            5  0.477212  0.197948   
4            6  0.425934  0.113676   
5            7  0.332772  0.246827   
6            8  0.303965  0.090546   
7           10  0.290219  0.023138   
8           11  0.257133  0.121045   
9           12  0.254673  0.009613   
10          13  0.235035  0.080244   
11          14  0.229955  0.021852   
12          17  0.203828  0.040202   
13          19  0.192554  0.028449   
14          20  0.191634  0.004790   
15          21  0.184490  0.037992   
16          23  0.180608  0.010632   
17          25  0.179924  0.001897   

                                             equation  
0                                            10.65562  
1                                    log(pX_E3x5_Lr2)  
2                       log(pX_E3x5_Lr2 + p_pt_track)  
3                        abs(p_

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe2 in position 4095: unexpected end of data

In [3]:
print(model.equations_[['complexity', 'loss', 'equation']].to_string())

    complexity      loss                                                                                                                                                                          equation
0            1  1.135352                                                                                                                                                                          10.65562
1            2  0.849831                                                                                                                                                                  log(pX_E3x5_Lr2)
2            4  0.581674                                                                                                                                                     log(pX_E3x5_Lr2 + p_pt_track)
3            5  0.477212                                                                                                                                                      abs(p_eta) + l

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe2 in position 4095: unexpected end of data

In [5]:
def predict_phys(X, a, b, c):
    return a * X['p_pt_track'] + b * X['pX_ecore'] / (X['p_ptcone40'] + c)

a, b, c = 0.72298244, 0.83139571, 1.26082979

y_pred_val  = predict_phys(X_val,  a, b, c)
y_pred_test = predict_phys(X_test, a, b, c)

print(f"RelMAD (val):  {relmad(y_val,  y_pred_val):.5f}")
print(f"RelMAD (test): {relmad(y_test, y_pred_test):.5f}")

RelMAD (val):  0.49354
RelMAD (test): 0.49936
Error in callback _flush_stdio (for post_execute), with arguments args (),kwargs {}:


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe2 in position 4095: unexpected end of data

In [2]:
# ----- PySR on full training data --------------------------------------------
X_pysr = X_train[top_features].values
y_pysr = np.log(y_train.values)
print(f"PySR fitting on {len(X_pysr)} events × {len(top_features)} features")
print(f"Target: log(E_true) in range [{y_pysr.min():.2f}, {y_pysr.max():.2f}]")

# ----- PySR setup -------------------------------------------------------------
# Restart kernel first, then:
model = PySRRegressor(
    niterations=200,         # fewer iterations
    populations=20,          # smaller population
    population_size=33,
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["log", "exp", "sqrt", "square", "abs"],
    maxsize=25,
    parsimony=0.0032,
    elementwise_loss="loss(prediction, target) = (prediction - target)^2",
    procs=4,
    model_selection="best",
    progress=False,         # disable the Unicode-prone progress display
    random_state=42,
    deterministic=False,
    verbosity=1,
    warm_start=False,       # ensure fresh start
)
model.fit(X_pysr, y_pysr, variable_names=top_features)

# ----- Inspect the Pareto front -----------------------------------------------
print("\n=== Pareto front ===")
print(model.equations_[['complexity', 'loss', 'score', 'equation']].to_string())

print("\n=== Best equation (symbolic) ===")
print(model.sympy())

# ----- Evaluate on full val/test ----------------------------------------------
def relmad(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_pred - y_true) / y_true))

# Predict log(E) on full val/test, then exp() to recover linear E
log_pred_val  = model.predict(X_val[top_features].values)
log_pred_test = model.predict(X_test[top_features].values)

y_pred_val  = np.exp(log_pred_val)
y_pred_test = np.exp(log_pred_test)

print(f"\nRelMAD (val,  best equation):  {relmad(y_val,  y_pred_val):.5f}")
print(f"RelMAD (test, best equation):  {relmad(y_test, y_pred_test):.5f}")

# Also evaluate every equation on the Pareto front for the writeup
print("\n=== RelMAD per equation on Pareto front ===")
for i, eq in enumerate(model.equations_.itertuples()):
    log_pred = model.predict(X_val[top_features].values, index=i)
    rm = relmad(y_val, np.exp(log_pred))
    print(f"  complexity={eq.complexity:2d}  loss={eq.loss:.4f}  "
          f"RelMAD={rm:.4f}  eq: {eq.equation}")

/Users/prometheus/Documents/Python/uni-env-3.13/lib/python3.13/site-packages/pysr/sr.py:2265: UserWarning: Note: you are running with more than 10,000 datapoints. You should consider turning on batching (https://ai.damtp.cam.ac.uk/pysr/options/#batching). You should also reconsider if you need that many datapoints. Unless you have a large amount of noise (in which case you should smooth your dataset first), generally < 10,000 datapoints is enough to find a functional form with symbolic regression. More datapoints will lower the search speed.
  warnings.warn(
Compiling Julia backend...


PySR fitting on 26460 events × 6 features
Target: log(E_true) in range [6.35, 13.68]


/Users/prometheus/Documents/Python/uni-env-3.13/lib/python3.13/site-packages/pysr/sr.py:1873: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(
[ Info: Note: you are running with more than 10,000 datapoints. You should consider turning on batching (`options.batching`), and also if you need that many datapoints. Unless you have a large amount of noise (in which case you should smooth your dataset first), generally < 10,000 datapoints is enough to find a functional form.
[ Info: Started!



Expressions evaluated per second: 1.510e+04
Progress: 62 / 4000 total iterations (1.550%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           1.102e+00  0.000e+00  y = 10.651
2           8.098e-01  3.083e-01  y = log(pX_E3x5_Lr2)
4           5.765e-01  1.699e-01  y = log(pX_E3x5_Lr2 + p_pt_track)
6           4.764e-01  9.539e-02  y = log(pX_E3x5_Lr2 + p_pt_track) + -0.31648
7           3.519e-01  3.028e-01  y = log(p_pt_track) + abs(p_eta / 1.3929)
10          3.519e-01  3.298e-06  y = log(p_pt_track) + abs(abs((p_eta / 1.3929) + -0.001961...
                                      5))
16          3.461e-01  2.793e-03  y = 0.63816 - (((log(p_pt_track) / -1.7372) + -0.19809) + ...
                                      (log(2.0657 / sqrt(pX_E3x5_Lr2)) + 0.31039))
─────────────────────────

[ Info: Final population:
[ Info: Results saved to:



=== Pareto front ===
    complexity      loss     score                                                                                                                                                                              equation
0            1  1.102208  0.000000                                                                                                                                                                             10.650813
1            2  0.809812  0.308269                                                                                                                                                                      log(pX_E3x5_Lr2)
2            4  0.576501  0.169913                                                                                                                                                         log(pX_E3x5_Lr2 + p_pt_track)
3            5  0.499952  0.142465                                                                            

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe2 in position 4095: unexpected end of data

In [3]:
# ----- Evaluate on full val/test ----------------------------------------------
def relmad(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_pred - y_true) / y_true))

# Predict log(E) on full val/test, then exp() to recover linear E
log_pred_val  = model.predict(X_val[top_features].values)
log_pred_test = model.predict(X_test[top_features].values)

y_pred_val  = np.exp(log_pred_val)
y_pred_test = np.exp(log_pred_test)

print(f"\nRelMAD (val,  best equation):  {relmad(y_val,  y_pred_val):.5f}")
print(f"RelMAD (test, best equation):  {relmad(y_test, y_pred_test):.5f}")

# Also evaluate every equation on the Pareto front for the writeup
print("\n=== RelMAD per equation on Pareto front ===")
for i, eq in enumerate(model.equations_.itertuples()):
    log_pred = model.predict(X_val[top_features].values, index=i)
    rm = relmad(y_val, np.exp(log_pred))
    print(f"  complexity={eq.complexity:2d}  loss={eq.loss:.4f}  "
          f"RelMAD={rm:.4f}  eq: {eq.equation}")


RelMAD (val,  best equation):  0.55579
RelMAD (test, best equation):  0.55654

=== RelMAD per equation on Pareto front ===
  complexity= 1  loss=1.1022  RelMAD=1.5721  eq: 10.650813
  complexity= 2  loss=0.8098  RelMAD=1.0233  eq: log(pX_E3x5_Lr2)
  complexity= 4  loss=0.5765  RelMAD=1.1286  eq: log(pX_E3x5_Lr2 + p_pt_track)
  complexity= 5  loss=0.5000  RelMAD=1.1165  eq: log(p_pt_track) + abs(p_eta)
  complexity= 6  loss=0.4342  RelMAD=0.6812  eq: log((pX_E3x5_Lr2 / 1.9893216) + p_pt_track)
  complexity= 7  loss=0.3519  RelMAD=0.5912  eq: log(p_pt_track) + abs(p_eta / 1.39285)
  complexity= 8  loss=0.2961  RelMAD=0.5558  eq: log(p_pt_track + (pX_ecore / (p_ptcone40 - -1.907946)))
  complexity= 9  loss=0.2961  RelMAD=0.5558  eq: log(abs(pX_ecore / (-1.9078947 - p_ptcone40)) + p_pt_track)
  complexity=10  loss=0.2831  RelMAD=0.4953  eq: log(p_pt_track + (pX_ecore / (p_ptcone40 - -1.1804906))) - 0.27125388
  complexity=11  loss=0.2627  RelMAD=0.5208  eq: log((pX_E3x5_Lr2 / (p_ptcone40 

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe2 in position 4095: unexpected end of data